# Step 5.1: Environment Setup and Hardware Configuration

## 1. Motivation for Transfer Learning with Xception

The previous experiments (Steps 2, 3 and 4) trained custom CNNs from scratch on the WikiArt dataset, revealing a consistent pattern: the models struggled to extract sufficiently rich visual representations within a reasonable training budget, stabilising at moderate F1-scores despite various regularisation efforts. On this notebook, instead of building feature extractors from random initialisations, we use transfer learning to inherit deep, well-generalised representations pretrained on ImageNet.

Xception is the natural first candidate for this transition. It is a well-established, widely benchmarked architecture whose depthwise separable convolution design achieves strong accuracy-to-parameter ratios whilst remaining computationally tractable on a consumer GPU. Its 299×299 native resolution (approximately 2.9× more pixels per image than the 128×128 used in earlier runs) should preserve finer brushstroke and texture detail that is central to distinguishing art styles. The goal of this notebook is to establish a solid transfer learning baseline before exploring more recent architectures in subsequent steps.

In [1]:
import os
from pathlib import Path
import json
import math
from keras import Model, layers
from keras.applications import Xception, xception
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler, EarlyStopping
from keras.backend import clear_session
from keras.utils import image_dataset_from_directory

## 2. Hardware Optimisation

Suppresses non-critical TensorFlow C++ runtime warnings by setting `TF_CPP_MIN_LOG_LEVEL` to `2` before importing TensorFlow, keeping training logs readable. Implements `set_memory_growth` to prevent TensorFlow from reserving the entirety of the VRAM at startup. Without this, the operating system and other processes may be starved of GPU memory, causing hard crashes.
XLA JIT compilation is defined but left commented out, as Xception's dynamic graph introduces tracing overhead that outweighs the benefit at this batch size.

In [2]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import tensorflow_addons as tfa

# ── GPU: memory growth ────────────────────────────────────────────────────────
# Prevents TF from reserving all VRAM at startup.
# Without this, the OS and browser might not be able to get GPU memory, in which case
# you get hard crashes.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detected: {[g.name for g in gpus]}")
else:
    print("No GPU — running on CPU.")

# ── XLA JIT compilation ───────────────────────────────────────────────────────
# Fuses TF ops into optimised GPU kernels.
# Adds a one-time ~30-60s compilation cost on the first batch, then speeds up
# all subsequent batches. Worth it for multi-epoch training.
# tf.config.optimizer.set_jit(True)
# print("XLA JIT enabled.")

GPU detected: ['/physical_device:GPU:0']


# Step 5.2: Transfer Learning Architecture - Xception

## 1. Model Design Philosophy

Defines `TransferXception`, a Keras `Model` subclass wrapping the pretrained Xception backbone with a lightweight classification head. Xception does not apply internal rescaling, it expects inputs in the range [-1, 1]. Rather than inserting a `Rescaling` layer into the model graph, preprocessing is applied inside the `call()` method via `xception.preprocess_input`, keeping the model graph clean and compatible with the externally augmented `tf.data` pipeline. Augmentation itself is handled entirely via Albumentations in the dataset pipeline.

## 2. Two-Phase Trainability Design

The constructor freezes `self.base` entirely at initialisation, enabling a clean Phase 1 where only the newly added head (Global Average Pooling, Dropout, and a softmax Dense layer) is updated against the ImageNet-pretrained representations. The `unfreeze_base` method implements Phase 2: it sets the base to trainable and then iterates over its layers, re-freezing the first `n_freeze` layers (defaulting to 115 out of Xception's ~134 total layers, preserving all low-level and mid-level feature detectors) and unconditionally re-freezing all Batch Normalisation layers throughout the network to prevent running-statistic drift from destabilising training on the art domain. A diagnostic print reports the frozen/unfrozen split so the configuration can be verified at runtime.

## 3. Configuration Serialisation

The `get_config` method extends the parent class configuration with the model's custom constructor arguments (`num_classes`, `dropout_rate`), ensuring the model can be correctly reconstructed from a saved checkpoint without manual argument tracking.

In [3]:
class TransferXception(Model):
    """
    Pre-trained Xception.
    Xception does NOT include internal rescaling — inputs must be in [-1, 1].
    We use xception.preprocess_input (maps [0,255] -> [-1,1]) directly on inputs.
    Augmentation is handled externally via tf.data.Dataset (Albumentations).
    """

    def __init__(self, num_classes, dropout_rate=0.5, **kwargs):
        super().__init__(**kwargs, name="transfer_xception")
        self.num_classes = num_classes
        self.dropout_rate = dropout_rate

        self.base = Xception(
            include_top=False,
            weights="imagenet"
        )
        self.base.trainable = False

        self.gap_layer = layers.GlobalAveragePooling2D()
        self.dropout_layer = layers.Dropout(dropout_rate)
        self.dense_layer = layers.Dense(self.num_classes, activation="softmax")

    def unfreeze_base(self, n_freeze=115):
        """
        Phase 2: unfreeze the top layers of the Xception base.
        Xception has ~134 layers — freezing the first 30 preserves low-level features.
        """
        self.base.trainable = True
        for i, layer in enumerate(self.base.layers):
            # RULE A: Freeze the first N layers (low-level features)
            if i < n_freeze:
                layer.trainable = False
            
            # RULE B: Freeze ALL Batch Normalization layers (for Stability)
            elif isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
        frozen = sum(1 for l in self.base.layers if not l.trainable)
        total  = len(self.base.layers)
        print(f"{self.name}: {frozen}/{total} base layers frozen, {total - frozen} unfrozen")

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_classes": self.num_classes,
            "dropout_rate": self.dropout_rate,
        })
        return config

    def call(self, inputs, training=False):
        # Step 1: preprocess to [-1, 1] as Xception expects
        x = xception.preprocess_input(inputs)

        # Step 2: forward through base
        x = self.base(x, training=training)

        x = self.gap_layer(x)
        x = self.dropout_layer(x, training=training)
        return self.dense_layer(x)

# Step 5.3: Hyperparameters and Data Pipeline

## 1. Global Configuration

Establishes the critical training parameters for both phases. The image resolution is set to 299×299, the native resolution for Xception. Resizing to any other value would require the model to generalise spatially in ways it was not trained for, degrading the quality of the pretrained features. The batch size of 16 balances GPU utilisation against memory constraints; reduce to 8 if OOM errors occur on a 8 GB GPU. Two distinct learning rates are defined: `PHASE1_LR` at 1e-3 for warm head training and `PHASE2_LR` at 1e-5 (approximately 100× lower) to fine-tune the unfrozen top layers without overwriting the pretrained representations. A fixed seed of 123 ensures reproducibility across dataset shuffling and augmentation. Directories for checkpoints and training metrics are created idempotently.

## 2. Dataset Instantiation and Mixup

Loads the train, validation, and test splits from the partitioned `wikiart_split` directory using `image_dataset_from_directory`, applying `crop_to_aspect_ratio=True` to avoid distortion when resizing to the square target resolution. Categorical label encoding is used throughout to match the softmax output format.

A `mixup` function with `alpha=0.4` is defined to blend pairs of images and their labels proportionally, acting as a data-space regulariser that discourages overconfident predictions. It is applied to the training dataset via `map()` with `AUTOTUNE`-parallelised execution, followed by prefetching to overlap preprocessing with GPU computation. The validation and test datasets are left unaugmented to ensure evaluation reflects true generalisation.

In [4]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
# 299×299: native resolution for Xception (significant accuracy gain over 224)
# Note: ~2.9× more pixels per image — reduce batch_size if you hit OOM on GPU
IMAGE_SIZE     = (299, 299)
BATCH_SIZE     = 16       # adjust based on your GPU's VRAM (e.g., 8 or 16 for 8GB, 32+ for 16GB)
PHASE1_EPOCHS  = 25       # frozen-base head training
PHASE2_EPOCHS  = 40       # fine-tuning (EarlyStopping will cut this short)
PHASE1_LR      = 1e-3     # higher LR — only head is updating
PHASE2_LR      = 1e-5     # ~100× lower LR — prevent destroying pretrained weights
N_CLASSES      = 23

data_dir_path = Path("..\wikiart_split")
root_dir_path = Path(".")
checkpoints_folder_path = root_dir_path / "Checkpoints"
if not os.path.exists(checkpoints_folder_path):
    os.makedirs(checkpoints_folder_path)
metrics_folder_path = root_dir_path / "Metrics"
if not os.path.exists(metrics_folder_path):
    os.makedirs(metrics_folder_path)

seed = 123

# ── Dataset loading ──────────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

# 1. Load raw images (batched) from directories
train_ds = image_dataset_from_directory(
    data_dir_path / "train",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=True,
    seed=seed,
)
val_ds = image_dataset_from_directory(
    data_dir_path / "val",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)
test_ds = image_dataset_from_directory(
    data_dir_path / "test",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)

# ── Mixup ────────────────────────────────────────────────────────────────────
# Blends pairs of images and their labels proportionally.
def mixup(images, labels, alpha=0.4):
    images = tf.cast(images, tf.float32)
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([], 0.0, alpha)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * images + (1.0 - lam) * tf.gather(images, indices)
    mixed_labels = lam * labels + (1.0 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels

# Applies mixup augmentation to the training dataset.
# We use map() to apply the mixup function to each batch of images and labels.
# The num_parallel_calls=AUTOTUNE argument allows TensorFlow to determine the optimal number of parallel calls for performance.
# Finally, we call prefetch(AUTOTUNE) to allow the dataset to fetch batches in the background while the model is training, improving performance.
train_ds_mixed = train_ds.map(mixup, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

Found 9326 files belonging to 23 classes.
Found 1992 files belonging to 23 classes.
Found 2022 files belonging to 23 classes.


# Step 5.4: Class Weights, Model Instantiation, and Metrics

## 1. Class Weights

Loads the pre-computed class weights from `class_weights.json`. These weights compensate for the significant class imbalance in the WikiArt dataset by scaling the loss contribution of under-represented art styles upward, preventing the model from achieving low training loss simply by predicting the majority classes.

In [5]:
# Load class weights
with open('..\class_weights.json', 'r') as f:
    class_weights = json.load(f)
class_weights = {int(k): v for k, v in class_weights.items()}


## 2. Model Instantiation

Calls `clear_session()` before instantiating the model to flush any residual graph state and variable allocations from previous runs in the same kernel session, freeing GPU memory cleanly. The model is instantiated with the 23-class output configuration at the default dropout rate of 0.5.

In [6]:
clear_session() # Clear previous models from memory before instantiating new ones.

model = TransferXception(num_classes=N_CLASSES)

## 3. Metrics and Loss

Defines a `make_metrics` factory function that returns a fresh set of stateful metric instances each time it is called. This is necessary because Keras metrics accumulate state across batches, and reusing the same instances across compilation calls would contaminate Phase 2 evaluation with Phase 1 statistics. The metric set comprises Categorical Accuracy, AUC (multi-label formulation), and Macro F1-score via `tfa.metrics.F1Score`, providing a balanced assessment across all 23 art style classes.

In [7]:
def make_metrics(num_classes):
    """Return a fresh set of metric instances (metrics are stateful — each model needs its own)."""
    return [
        CategoricalAccuracy(name="accuracy"),
        AUC(multi_label=True, name="auc"),
        tfa.metrics.F1Score(num_classes=num_classes, average="macro", name="f1_score")
    ]


# Step 5.5: Learning Rate Schedule

## 1. Cosine Annealing with Linear Warmup

Defines a `make_cosine_warmup_scheduler` factory that returns a `LearningRateScheduler`-compatible function combining two regimes. During the warmup phase (the first `warmup_epochs` epochs), the learning rate rises linearly from 0 to `base_lr`. This prevents the randomly initialised classification head from producing large gradients that could damage the pretrained Xception features before the head has learned to produce reasonable outputs. After warmup, the learning rate follows a cosine decay curve from `base_lr` down to approximately 0, which tends to find better minima than step-decay or exponential schedules by smoothly annealing the optimisation landscape rather than abruptly reducing the step size.

In [8]:
def make_cosine_warmup_scheduler(base_lr, total_epochs, warmup_epochs=5):
    """
    Cosine annealing with linear warmup.

    Warmup: LR ramps linearly from 0 to base_lr over the first warmup_epochs.
    This prevents the randomly initialised head from producing large gradients
    that destabilise the pretrained base at the start of training.

    Cosine decay: LR then follows a cosine curve from base_lr down to ~0.
    Finds better minima than step-decay or exponential decay in practice.
    """
    def scheduler(epoch, lr):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))
    return scheduler

# Step 5.6: Phase 1 - Train Head with Frozen Base

## 1. Head Training Strategy

In Phase 1 the pretrained Xception base is completely frozen and only the GAP + Dropout + Dense head is updated. This is the standard transfer learning warm-up: the head is randomly initialised and would otherwise produce destructively large gradients if the base were simultaneously trainable. By isolating the head update for the first `PHASE1_EPOCHS` epochs, the model learns to map Xception's ImageNet features to the 23 WikiArt classes without disturbing the pretrained representations.

## 2. Compilation and Callbacks

Compiled with `AdamW` at `PHASE1_LR` and a weight decay of 1e-6, `CategoricalCrossentropy` with `label_smoothing=0.1` to discourage overconfident softmax outputs, and fresh metrics from `make_metrics`. The callback stack saves the best checkpoint by validation loss, logs per-epoch metrics to CSV, applies the cosine warmup schedule with 3 warmup epochs, and stops early with a patience of 5 epochs to avoid wasting compute if the head converges prematurely.

In [9]:
print(f"\n{'='*60}")
print(f"Phase 1 training: {model.name}")
print(f"{'='*60}")


model.compile(
    optimizer=tfa.optimizers.AdamW(learning_rate=PHASE1_LR, weight_decay=1e-6),
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1),
    metrics=make_metrics(num_classes=N_CLASSES),
)

callbacks = [
    ModelCheckpoint(
        checkpoints_folder_path / f"ckpt_phase1_{model.name}.tf",
        monitor="val_loss", save_best_only=True, verbose=1,
    ),
    CSVLogger(metrics_folder_path / f"log_phase1_{model.name}.csv"),
    LearningRateScheduler(
        make_cosine_warmup_scheduler(PHASE1_LR, PHASE1_EPOCHS, warmup_epochs=3)
    ),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_ds_mixed,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)
phase1_fit_data = history

print("\nPhase 1 complete.")


Phase 1 training: transfer_xception
Epoch 1/25
583/583 [==============================] - ETA: 0s - loss: 2.8226 - accuracy: 0.2521 - auc: 0.6580 - f1_score: 0.2039
Epoch 1: val_loss improved from inf to 2.29657, saving model to Checkpoints\ckpt_phase1_transfer_xception.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_xception.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_xception.tf\assets


583/583 [==============================] - 102s 160ms/step - loss: 2.8226 - accuracy: 0.2521 - auc: 0.6580 - f1_score: 0.2039 - val_loss: 2.2966 - val_accuracy: 0.4618 - val_auc: 0.8986 - val_f1_score: 0.4242 - lr: 3.3333e-04
Epoch 2/25
583/583 [==============================] - ETA: 0s - loss: 2.4564 - accuracy: 0.4213 - auc: 0.7263 - f1_score: 0.3433
Epoch 2: val_loss improved from 2.29657 to 1.98359, saving model to Checkpoints\ckpt_phase1_transfer_xception.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_xception.tf\assets


...

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_xception.tf\assets


583/583 [==============================] - 92s 158ms/step - loss: 2.0845 - accuracy: 0.5918 - auc: 0.7693 - f1_score: 0.5041 - val_loss: 1.6457 - val_accuracy: 0.6566 - val_auc: 0.9547 - val_f1_score: 0.6360 - lr: 3.5913e-04
Epoch 18/25
583/583 [==============================] - ETA: 0s - loss: 2.0913 - accuracy: 0.6027 - auc: 0.7664 - f1_score: 0.5109
Epoch 18: val_loss did not improve from 1.64565
583/583 [==============================] - 76s 130ms/step - loss: 2.0913 - accuracy: 0.6027 - auc: 0.7664 - f1_score: 0.5109 - val_loss: 1.6517 - val_accuracy: 0.6556 - val_auc: 0.9550 - val_f1_score: 0.6373 - lr: 2.9229e-04
Epoch 19/25
583/583 [==============================] - ETA: 0s - loss: 2.1025 - accuracy: 0.5895 - auc: 0.7711 - f1_score: 0.4949
Epoch 19: val_loss improved from 1.64565 to 1.63550, saving model to Checkpoints\ckpt_phase1_transfer_xception.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_xception.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_xception.tf\assets


583/583 [==============================] - 92s 157ms/step - loss: 2.1025 - accuracy: 0.5895 - auc: 0.7711 - f1_score: 0.4949 - val_loss: 1.6355 - val_accuracy: 0.6697 - val_auc: 0.9551 - val_f1_score: 0.6494 - lr: 2.2968e-04
Epoch 20/25
583/583 [==============================] - ETA: 0s - loss: 2.1154 - accuracy: 0.5861 - auc: 0.7727 - f1_score: 0.4919
Epoch 20: val_loss did not improve from 1.63550
583/583 [==============================] - 76s 130ms/step - loss: 2.1154 - accuracy: 0.5861 - auc: 0.7727 - f1_score: 0.4919 - val_loss: 1.6425 - val_accuracy: 0.6601 - val_auc: 0.9554 - val_f1_score: 0.6398 - lr: 1.7257e-04
Epoch 21/25
583/583 [==============================] - ETA: 0s - loss: 2.0822 - accuracy: 0.6017 - auc: 0.7739 - f1_score: 0.5073
Epoch 21: val_loss did not improve from 1.63550
583/583 [==============================] - 76s 130ms/step - loss: 2.0822 - accuracy: 0.6017 - auc: 0.7739 - f1_score: 0.5073 - val_loss: 1.6404 - val_accuracy: 0.6657 - val_auc: 0.9554 - val_f1_

# Step 5.7: Phase 1 Evaluation

## 1. Baseline Performance

Evaluates the Phase 1 checkpoint against the held-out test set to establish a clean baseline before the Xception base is unfrozen. This score reflects how well the frozen ImageNet representations alone, combined with the newly trained head, generalise to unseen WikiArt images, and serves as the lower bound against which Phase 2 fine-tuning will be assessed.

In [10]:
phase1_eval_data = model.evaluate(
    test_ds,
    batch_size=BATCH_SIZE,
    return_dict=True,
    verbose=0
)
phase1_eval_data

{'loss': 1.635571837425232,
 'accuracy': 0.6617210507392883,
 'auc': 0.9551092982292175,
 'f1_score': 0.6452075839042664}

# Step 5.8: Phase 2 - Fine-tune Unfrozen Base Layers

# 1. Fine-tuning Strategy

Phase 2 unfreezes the top layers of the Xception base and retrains the entire unfrozen network at a much lower learning rate. The underlying intuition is that Xception's early layers capture generic low-level features (edges, colour gradients, and simple textures) that transfer well across visual domains and should remain frozen, whilst the later layers encode higher-level semantic patterns more amenable to adaptation towards the stylistic vocabulary of the WikiArt dataset. With a default `n_freeze` of 115, only the final ~19 layers are released for fine-tuning, keeping the intervention minimal and controlled.

## 2. Recompilation and Callbacks

The base is unfrozen via `model.unfreeze_base()`. The model is recompiled at `PHASE2_LR` (1e-5) with a weight decay of 1e-7 to apply very gentle L2-style regularisation without overwriting the pretrained representations. EarlyStopping patience is increased to 10 epochs; Improvements during fine-tuning are smaller and more gradual than during head training, so more patience is needed to distinguish genuine plateaus from temporary fluctuations introduced by the low learning rate.


In [11]:
print(f"\n{'='*60}")
print(f"Phase 2 fine-tuning: {model.name}")
print(f"{'='*60}")

# Unfreeze top layers — defaults are set inside each model class
model.unfreeze_base()

# Recompile at ~100× lower LR to avoid overwriting pretrained representations
model.compile(
    optimizer=tfa.optimizers.AdamW(learning_rate=PHASE2_LR, weight_decay=1e-7),
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1),
    metrics=make_metrics(num_classes=N_CLASSES),
)

callbacks = [
    ModelCheckpoint(
        checkpoints_folder_path / f"ckpt_phase2_{model.name}.tf",
        monitor="val_loss", save_best_only=True, verbose=1,
    ),
    CSVLogger(metrics_folder_path / f"log_phase2_{model.name}.csv"),
    LearningRateScheduler(
        make_cosine_warmup_scheduler(PHASE2_LR, PHASE2_EPOCHS, warmup_epochs=2)
    ),
    # More patience in Phase 2 — improvements are smaller and slower
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_ds_mixed,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)
phase2_fit_data = history

print("\nPhase 2 complete.")



Phase 2 fine-tuning: transfer_xception
transfer_xception: 120/132 base layers frozen, 12 unfrozen
Epoch 1/40
583/583 [==============================] - ETA: 0s - loss: 2.0622 - accuracy: 0.6204 - auc: 0.7735 - f1_score: 0.5229
Epoch 1: val_loss improved from inf to 1.59828, saving model to Checkpoints\ckpt_phase2_transfer_xception.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_xception.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_xception.tf\assets


583/583 [==============================] - 111s 183ms/step - loss: 2.0622 - accuracy: 0.6204 - auc: 0.7735 - f1_score: 0.5229 - val_loss: 1.5983 - val_accuracy: 0.6757 - val_auc: 0.9582 - val_f1_score: 0.6538 - lr: 5.0000e-06
Epoch 2/40
583/583 [==============================] - ETA: 0s - loss: 2.0158 - accuracy: 0.6366 - auc: 0.7748 - f1_score: 0.5386
Epoch 2: val_loss improved from 1.59828 to 1.56051, saving model to Checkpoints\ckpt_phase2_transfer_xception.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_xception.tf\assets


...

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_xception.tf\assets


583/583 [==============================] - 105s 180ms/step - loss: 1.7116 - accuracy: 0.7989 - auc: 0.8008 - f1_score: 0.6790 - val_loss: 1.3240 - val_accuracy: 0.7701 - val_auc: 0.9773 - val_f1_score: 0.7476 - lr: 1.3214e-06
Epoch 33/40
583/583 [==============================] - ETA: 0s - loss: 1.7217 - accuracy: 0.7920 - auc: 0.8042 - f1_score: 0.6715
Epoch 33: val_loss did not improve from 1.32400
583/583 [==============================] - 89s 153ms/step - loss: 1.7217 - accuracy: 0.7920 - auc: 0.8042 - f1_score: 0.6715 - val_loss: 1.3241 - val_accuracy: 0.7731 - val_auc: 0.9773 - val_f1_score: 0.7516 - lr: 1.0543e-06
Epoch 34/40
583/583 [==============================] - ETA: 0s - loss: 1.7165 - accuracy: 0.7955 - auc: 0.8017 - f1_score: 0.6750
Epoch 34: val_loss did not improve from 1.32400
583/583 [==============================] - 90s 154ms/step - loss: 1.7165 - accuracy: 0.7955 - auc: 0.8017 - f1_score: 0.6750 - val_loss: 1.3242 - val_accuracy: 0.7691 - val_auc: 0.9772 - val_f1

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_xception.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_xception.tf\assets


583/583 [==============================] - 106s 181ms/step - loss: 1.7023 - accuracy: 0.7948 - auc: 0.8061 - f1_score: 0.6764 - val_loss: 1.3206 - val_accuracy: 0.7711 - val_auc: 0.9775 - val_f1_score: 0.7496 - lr: 4.2113e-07
Epoch 37/40
583/583 [==============================] - ETA: 0s - loss: 1.7422 - accuracy: 0.7898 - auc: 0.8036 - f1_score: 0.6648
Epoch 37: val_loss did not improve from 1.32058
583/583 [==============================] - 89s 153ms/step - loss: 1.7422 - accuracy: 0.7898 - auc: 0.8036 - f1_score: 0.6648 - val_loss: 1.3241 - val_accuracy: 0.7716 - val_auc: 0.9773 - val_f1_score: 0.7498 - lr: 2.7091e-07
Epoch 38/40
583/583 [==============================] - ETA: 0s - loss: 1.7034 - accuracy: 0.8017 - auc: 0.7957 - f1_score: 0.6839
Epoch 38: val_loss did not improve from 1.32058
583/583 [==============================] - 90s 154ms/step - loss: 1.7034 - accuracy: 0.8017 - auc: 0.7957 - f1_score: 0.6839 - val_loss: 1.3225 - val_accuracy: 0.7721 - val_auc: 0.9774 - val_f1

# Step 5.9: Phase 2 Evaluation and Final Results

## 1. Final Performance Assessment

Evaluates the best Phase 2 checkpoint against the held-out test set to produce the definitive performance figures for the Xception transfer learning baseline. Comparing these metrics against the Phase 1 baseline quantifies the gain attributable to fine-tuning the top layers on the art domain. These results will serve as the direct reference point when evaluating more recent architectures in the following step.

In [12]:
phase2_eval_data = model.evaluate(
    test_ds,
    batch_size=BATCH_SIZE,
    return_dict=True,
    verbose=0
)
phase2_eval_data

{'loss': 1.3174922466278076,
 'accuracy': 0.7764589786529541,
 'auc': 0.9773527979850769,
 'f1_score': 0.7659110426902771}